# Config

In [1]:
"""
Project configuration.

"""

from pathlib import Path

if "__file__" in globals():
    ROOT = Path(__file__).resolve().parent.parent
else:
    ROOT = Path.cwd()
    

DATA_DIR = ROOT / "data" / "raw"
ASSETS_DIR = ROOT / "assets"

OUTPUT_DIR = ROOT / "outputs"
FRAME_DIR = OUTPUT_DIR / "frames"
GIF_DIR = OUTPUT_DIR / "gifs"

FRAME_DIR.mkdir(parents=True, exist_ok=True)
GIF_DIR.mkdir(parents=True, exist_ok=True)

SHAPEFILE = "MG_Municipios_2021.shp"

CSV_FILES = {
    2018: "MG DADOS ELEICAO 2018.csv",
    2022: "MG DADOS ELEICAO 2022.csv",
}

GRID_RESOLUTION = 300
IDW_POWER = 2
FIG_DPI = 300
FRAME_DURATION = 2200

# ELECTION METADATA

ELECTIONS = [
    {
        "year": 2018,
        "party": "PT",
        "candidate": "Fernando Pimentel",
        "column": "% 13",
        "photo": "pimentel.jpg",
    },
    {
        "year": 2018,
        "party": "PSDB",
        "candidate": "Antonio Anastasia",
        "column": "% 45",
        "photo": "anastasia.jpg",
    },
    {
        "year": 2018,
        "party": "NOVO",
        "candidate": "Romeu Zema",
        "column": "% 30",
        "photo": "zema.jpg",
    },
    {
        "year": 2022,
        "party": "NOVO",
        "candidate": "Romeu Zema",
        "column": "% 30",
        "photo": "zema.jpg",
    },
    {
        "year": 2022,
        "party": "PSD",
        "candidate": "Alexandre Kalil",
        "column": "% 55",
        "photo": "kalil.jpg",
    },
    {
        "year": 2022,
        "party": "PL",
        "candidate": "Carlos Viana",
        "column": "% 22",
        "photo": "viana.jpg",
    },
]

# data_loader

In [2]:
"""
Data loading utilities.

"""

import pandas as pd
import geopandas as gpd

# LOAD MUNICIPAL BOUNDARIES

def load_municipalities():
    """
    Load Minas Gerais municipal boundaries.
    """

    gdf = gpd.read_file(
        DATA_DIR / SHAPEFILE
    )

    gdf = gdf.to_crs(
        epsg=31983
    )

    gdf["CD_MUN"] = (
        gdf["CD_MUN"]
        .astype(str)
        .str.zfill(7)
    )

    return gdf


# LOAD ELECTION RESULTS

def load_results(year):
    """
    Load election results.
    """

    if year not in CSV_FILES:
        raise ValueError(f"Unsupported year: {year}")

    df = pd.read_csv(
        DATA_DIR / CSV_FILES[year],
        encoding="latin1",
        sep=None,
        engine="python",
    )

    df.columns = df.columns.str.strip()

    df["CD_MUN"] = (
        df["CD_MUN"]
        .astype(str)
        .str.zfill(7)
    )

    return df



# LOAD COMPLETE DATASET

def load_dataset(year):
    """
    Merge municipal boundaries with election results.
    """

    municipalities = load_municipalities()
    results = load_results(year)

    gdf = municipalities.merge(
        results,
        on="CD_MUN",
        how="inner",
    )

    centroid = gdf.geometry.centroid
    gdf["x"] = centroid.x
    gdf["y"] = centroid.y

    return gdf

# interpolation

In [5]:
"""
Spatial interpolation utilities.

"""

import numpy as np
import geopandas as gpd
from shapely.geometry import Point


def create_grid(gdf, resolution=300):
    """
    Create a regular interpolation grid.
    """

    xmin, ymin, xmax, ymax = gdf.total_bounds

    grid_x, grid_y = np.meshgrid(
        np.linspace(xmin, xmax, resolution),
        np.linspace(ymin, ymax, resolution),
    )

    return grid_x, grid_y


def idw(
    x,
    y,
    values,
    grid_x,
    grid_y,
    power=2,
):
    """
Perform Inverse Distance Weighting (IDW) interpolation.

Parameters
----------
x, y : array-like
    Coordinates of the observed points.

values : array-like
    Values associated with the observed points.

grid_x, grid_y : ndarray
    Coordinates of the interpolation grid.

power : int, default=2
    Exponent controlling how quickly the influence of neighboring
    observations decreases with distance.

Returns
-------
ndarray
    Interpolated values over the target grid.
"""

    distance = np.sqrt(
        (grid_x[..., None] - x) ** 2 +
        (grid_y[..., None] - y) ** 2
    )

    distance[distance == 0] = 1e-12

    weights = 1 / distance ** power

    surface = (
        np.sum(weights * values, axis=2)
        /
        np.sum(weights, axis=2)
    )

    return surface


def build_mask(gdf, grid_x, grid_y):
    """
    Mask interpolation outside the state boundary.
    """

    points = [
        Point(x, y)
        for x, y in zip(
            grid_x.ravel(),
            grid_y.ravel(),
        )
    ]

    points = gpd.GeoDataFrame(
        geometry=points,
        crs=gdf.crs,
    )

    # alteração sugerida
    boundary = gdf.geometry.union_all()

    mask = (
        ~points
        .within(boundary)
        .values
        .reshape(grid_x.shape)
    )

    return mask


def interpolate(
    gdf,
    column,
    resolution=300,
    power=2,
):
    """
    Complete interpolation workflow.
    """

    grid_x, grid_y = create_grid(
        gdf,
        resolution,
    )

    surface = idw(
        gdf["x"].values,
        gdf["y"].values,
        gdf[column].astype(float).values,
        grid_x,
        grid_y,
        power,
    )

    mask = build_mask(
        gdf,
        grid_x,
        grid_y,
    )

    surface = np.ma.array(
        surface,
        mask=mask,
    )

    surface = np.clip(
        surface,
        0,
        100,
    )

    return grid_x, grid_y, surface 

# Plotting 

In [6]:
"""
Plotting utilities.

"""

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import numpy as np

from PIL import Image

from matplotlib.offsetbox import OffsetImage, AnnotationBbox


# PHOTO

def load_photo(photo):
    """
    Load candidate photograph.
    """

    path = ASSETS_DIR / photo

    if not path.exists():
        return None

    image = Image.open(path)
    image = image.convert("RGB")
    image = image.resize((300, 300))

    return OffsetImage(
        np.array(image),
        zoom=0.18,
    )


# HEADER

def add_header(
    ax,
    candidate,
    party,
    year,
    photo,
):
    """
    Draw figure header.
    """

    image = load_photo(photo)

    if image is not None:

        artist = AnnotationBbox(
            image,
            (0.10, 0.50),
            frameon=False,
            xycoords="axes fraction",
        )

        ax.add_artist(artist)

        circle = patches.Circle(
            (0.10, 0.50),
            0.085,
            transform=ax.transAxes,
            facecolor="none",
            edgecolor="none",
        )

        ax.add_patch(circle)

        artist.set_clip_path(circle)

    ax.text(
        0.20,
        0.72,
        f"Minas Gerais - Eleição para Governador • {year}",
        fontsize=10,
        color="gray",
        transform=ax.transAxes,
    )

    ax.text(
        0.20,
        0.20,
        f"{candidate} ({party})",
        fontsize=18,
        fontweight="bold",
        transform=ax.transAxes,
    )

    ax.axis("off")

In [7]:
# CHOROPLETH

def plot_choropleth(
    gdf,
    column,
    title=None,
    cmap="YlOrRd",
    figsize=(8, 8),
):

    fig, ax = plt.subplots(figsize=figsize)

    gdf.plot(
        column=column,
        cmap=cmap,
        linewidth=0.20,
        edgecolor="white",
        legend=True,
        ax=ax,
    )

    ax.set_axis_off()

    if title is not None:
        ax.set_title(title)

    plt.tight_layout()

    return fig


# IDW SURFACE

def plot_idw(
    gdf,
    surface,
    candidate,
    party,
    year,
    photo,
    output=None,
):

    xmin, ymin, xmax, ymax = gdf.total_bounds

    fig = plt.figure(figsize=(10, 11))

    gs = gridspec.GridSpec(
        2,
        1,
        height_ratios=[0.30, 8],
    )

    ax_header = fig.add_subplot(gs[0])

    ax = fig.add_subplot(gs[1])

    heat = ax.imshow(
        surface,
        extent=(xmin, xmax, ymin, ymax),
        origin="lower",
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
    )

    gdf.boundary.plot(
        ax=ax,
        linewidth=0.35,
        color="black",
        alpha=0.30,
    )

    ax.set_axis_off()

    add_header(
        ax_header,
        candidate,
        party,
        year,
        photo,
    )

    cbar = plt.colorbar(
        heat,
        ax=ax,
        fraction=0.03,
        pad=0.03,
    )

    cbar.set_label(
        f"Vote share (%) - {party}"
    )

    plt.subplots_adjust(
        top=0.98,
        bottom=0.03,
        left=0.02,
        right=0.92,
        hspace=-0.30,
    )

    if output is not None:

        plt.savefig(
            output,
            dpi=FIG_DPI,
            bbox_inches="tight",
        )

        plt.close(fig)

    else:

        return fig 

# Animation 

In [8]:
"""
Animation utilities.

"""

from PIL import Image


def create_animation():
    """
    Generate PNG frames and an animated GIF.
    """

    frame_files = []

    for election in ELECTIONS:

        print(
            f'Generating {election["candidate"]} ({election["year"]})...'
        )

        # Load election dataset

        gdf = load_dataset(
            election["year"]
        )

        # IDW interpolation

        _, _, surface = interpolate(
            gdf=gdf,
            column=election["column"],
            resolution=GRID_RESOLUTION,
            power=IDW_POWER,
        )

        
        # Output filename

        filename = (
            f'{election["year"]}_'
            f'{election["party"]}_'
            f'{election["candidate"].replace(" ", "_")}.png'
        )

        frame_path = FRAME_DIR / filename

        # Save frame
        

        plot_idw(
            gdf=gdf,
            surface=surface,
            candidate=election["candidate"],
            party=election["party"],
            year=election["year"],
            photo=election["photo"],
            output=frame_path,
        )

        frame_files.append(frame_path)

    # Create GIF
    

    images = [
        Image.open(frame).copy()
        for frame in frame_files
    ]

    gif_path = GIF_DIR / "mg_eleicao_governador_2018_2022.gif"

    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=FRAME_DURATION,
        loop=0,
    )

    print("\nAnimation successfully created.")
    print(gif_path)

    return gif_path 

# Main

In [9]:
"""
Animation utilities.

"""

from PIL import Image


def create_animation():
    """
    Generate PNG frames and compile them into an animated GIF.
    """

    frame_files = []

    # Generate PNG frames
    for election in ELECTIONS:

        print(
            f'Generating {election["candidate"]} ({election["year"]})...'
        )

        # Load election dataset
        gdf = load_dataset(
            election["year"]
        )

        # Perform IDW interpolation
        _, _, surface = interpolate(
            gdf=gdf,
            column=election["column"],
            resolution=GRID_RESOLUTION,
            power=IDW_POWER,
        )

        # Define output filename
        filename = (
            f'{election["year"]}_'
            f'{election["party"]}_'
            f'{election["candidate"].replace(" ", "_")}.png'
        )

        frame_path = FRAME_DIR / filename

        # Save frame
        plot_idw(
            gdf=gdf,
            surface=surface,
            candidate=election["candidate"],
            party=election["party"],
            year=election["year"],
            photo=election["photo"],
            output=frame_path,
        )

        frame_files.append(frame_path)

    # Assemble animated GIF
    images = [
        Image.open(frame).copy()
        for frame in frame_files
    ]

    gif_path = GIF_DIR / "mg_eleicao_governador_2018_2022.gif"

    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=FRAME_DURATION,
        loop=0,
    )

    print("\nAnimation successfully created.")
    print(gif_path)

    return gif_path

In [ ]:
"""
Main script.

"""

def main():
    create_animation()


if __name__ == "__main__":
    main()

Generating Fernando Pimentel (2018)...
Generating Antonio Anastasia (2018)...
